# Real Traffic Test — DNS Tunnel Classifier

Tests `dns1.pcapng` and `dns2.pcapng` against the trained model.

**Pipeline:**
1. Load pcapng → packet list (using training extractor)
2. Slide 10-second windows per source IP
3. Extract 38 features per window
4. Predict with `dns_tunnel_classifier.joblib`
5. Analyse results & flag CLI tool issues

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings('ignore')

REPO = Path('.')
PCAP1 = Path(r'C:\Users\acer\Downloads\dns1.pcapng')
PCAP2 = Path(r'C:\Users\acer\Downloads\dns2.pcapng')
MODEL_PATH = REPO / 'dns_tunnel_classifier.joblib'

print('Paths exist?')
for p in [PCAP1, PCAP2, MODEL_PATH]:
    print(f'  {p.name}: {p.exists()} ({p.stat().st_size/1024:.1f} KB)')

Paths exist?
  dns1.pcapng: True (455.1 KB)
  dns2.pcapng: True (69.0 KB)
  dns_tunnel_classifier.joblib: True (1307.9 KB)


## 1. Feature schema audit — CLI tool vs model

In [2]:
# Load model and inspect stored feature list
artifact = joblib.load(MODEL_PATH)

if isinstance(artifact, dict):
    model = artifact['model']
    model_features = artifact.get('features') or artifact.get('feature_names')
else:
    model = artifact
    model_features = list(getattr(model, 'feature_names_in_', []))

print(f'Model type : {type(model).__name__}')
print(f'Stored features ({len(model_features)}): {model_features}')

Model type : VotingClassifier
Stored features (36): ['n_packets', 'duration_sec', 'query_rate', 'unique_qnames', 'unique_subdomains', 'unique_qname_ratio', 'iat_mean', 'iat_std', 'iat_min', 'iat_max', 'payload_mean', 'payload_std', 'payload_max', 'entropy_mean', 'entropy_std', 'subdomain_entropy_mean', 'txt_frac', 'null_frac', 'aaaa_frac', 'a_frac', 'any_frac', 'tunnel_type_frac', 'tcp_frac', 'avg_qname_len', 'avg_subdomain_len', 'avg_label_count', 'avg_max_label_len', 'avg_b64_ratio', 'avg_hex_ratio', 'avg_numeric_ratio', 'avg_consonant_ratio', 'avg_unique_char_ratio', 'n_responses', 'avg_answer_count', 'avg_rdata_len', 'nxdomain_frac']


In [3]:
# CLI tool feature list (from DNS_tunnelling_CLI/feature_extraction.py)
CLI_FEATURES = [
    'n_packets', 'duration_sec', 'query_rate', 'unique_qnames', 'unique_subdomains',
    'unique_qname_ratio', 'iat_mean', 'iat_std', 'iat_min', 'iat_max',
    'payload_mean', 'payload_std', 'payload_max', 'entropy_mean', 'entropy_std',
    'subdomain_entropy_mean', 'txt_frac', 'null_frac', 'aaaa_frac', 'a_frac',
    'any_frac', 'tunnel_type_frac', 'tcp_frac', 'avg_qname_len', 'avg_subdomain_len',
    'avg_label_count', 'avg_max_label_len', 'avg_b64_ratio', 'avg_hex_ratio',
    'avg_numeric_ratio', 'avg_consonant_ratio', 'avg_unique_char_ratio',
    'n_responses', 'avg_answer_count', 'avg_rdata_len', 'nxdomain_frac',
]

model_set = set(model_features)
cli_set   = set(CLI_FEATURES)

missing_in_cli   = model_set - cli_set
extra_in_cli     = cli_set   - model_set

print('=== FEATURE MISMATCH AUDIT ===')
print(f'Model expects  : {len(model_features)} features')
print(f'CLI tool has   : {len(CLI_FEATURES)} features')
print()
print(f'In model but NOT in CLI : {sorted(missing_in_cli) or "none"}')
print(f'In CLI but NOT in model : {sorted(extra_in_cli)   or "none"}')

if missing_in_cli:
    print()
    print('*** CLI tool will CRASH on model load — inference.py raises ValueError ***')
    print('*** Fix: add missing features to EXPECTED_FEATURES and compute_features() ***')

=== FEATURE MISMATCH AUDIT ===
Model expects  : 36 features
CLI tool has   : 36 features

In model but NOT in CLI : none
In CLI but NOT in model : none


## 2. Load pcaps using the TRAINING extractor

We bypass the CLI tool extractor and call `feature_extract.py` directly — it produces the correct feature set.

In [4]:
# Make sure the training extractor is importable
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from feature_extract import (
    pcap_to_packet_list,
    sliding_windows,
    aggregate_window,
)

WINDOW_SEC  = 10.0
STRIDE_SEC  = 10.0   # non-overlapping (matches training)
MIN_PACKETS = 3      # relaxed — real captures may have fewer packets per window

def pcap_to_feature_rows(pcap_path: Path, label_hint: str) -> pd.DataFrame:
    """Read a pcap, slide 10-s windows per source IP, return feature DataFrame."""
    raw = pcap_to_packet_list(str(pcap_path))
    if not raw:
        print(f'  [WARN] No DNS packets found in {pcap_path.name}')
        return pd.DataFrame()

    print(f'{pcap_path.name}: {len(raw)} DNS packets')

    # Summary of IPs and query types
    from collections import Counter
    ips = Counter(p['src_ip'] for p in raw)
    qtypes = Counter(p['qtype'] for p in raw)
    print(f'  Source IPs   : {dict(ips.most_common(5))}')
    print(f'  Query types  : {dict(qtypes.most_common(8))}')
    t_span = max(p['timestamp'] for p in raw) - min(p['timestamp'] for p in raw)
    print(f'  Time span    : {t_span:.1f}s')
    print()

    # Group by source IP, then slide windows
    from collections import defaultdict
    by_ip = defaultdict(list)
    for p in raw:
        by_ip[p['src_ip']].append(p)

    rows = []
    for ip, pkts in by_ip.items():
        pkts.sort(key=lambda p: p['timestamp'])
        for row in sliding_windows(pkts, WINDOW_SEC, STRIDE_SEC, MIN_PACKETS):
            row['src_ip']   = ip
            row['pcap']     = pcap_path.name
            row['hint']     = label_hint
            rows.append(row)

    return pd.DataFrame(rows) if rows else pd.DataFrame()

df1 = pcap_to_feature_rows(PCAP1, label_hint='dns1')
df2 = pcap_to_feature_rows(PCAP2, label_hint='dns2')

df_all = pd.concat([df1, df2], ignore_index=True)
print(f'Total windows extracted: {len(df_all)}')

dns1.pcapng: 1270 DNS packets

  Source IPs   : {'127.0.0.1': 891, '127.0.0.53': 379}

  Query types  : {1: 962, 65: 277, 28: 31}

  Time span    : 384.0s

dns2.pcapng: 46 DNS packets

  Source IPs   : {'127.0.0.1': 26, '127.0.0.53': 20}

  Query types  : {1: 30, 65: 10, 28: 6}

  Time span    : 87.7s

Total windows extracted: 60

## 3. Run inference

In [5]:
FEATURE_COLS = [f for f in model_features if f in df_all.columns]
missing_cols = [f for f in model_features if f not in df_all.columns]

if missing_cols:
    print(f'[WARN] Columns missing from extracted data, filling with 0: {missing_cols}')
    for col in missing_cols:
        df_all[col] = 0.0

X = df_all[model_features].fillna(0).values

# Predict
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    proba  = model.predict_proba(X)

# Determine which column is TUNNEL (class 1)
classes = list(model.classes_)
tunnel_idx = classes.index(1) if 1 in classes else 1
benign_idx = 1 - tunnel_idx

df_all['p_tunnel']  = proba[:, tunnel_idx]
df_all['p_benign']  = proba[:, benign_idx]
df_all['decision']  = (df_all['p_tunnel'] >= 0.5).map({True: 'TUNNEL', False: 'BENIGN'})

# Per-base-learner probabilities
for name, est in model.named_estimators_.items():
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        bp = est.predict_proba(X)[:, tunnel_idx]
    df_all[f'p_tunnel_{name}'] = bp

print('Inference complete.')
print(df_all[['pcap', 'src_ip', 'window_start', 'n_packets', 'decision', 'p_tunnel']].to_string())

Inference complete.

           pcap      src_ip  window_start  n_packets decision  p_tunnel
0   dns1.pcapng   127.0.0.1  1.777906e+09         37   TUNNEL  0.736972
1   dns1.pcapng   127.0.0.1  1.777906e+09         47   BENIGN  0.342352
2   dns1.pcapng   127.0.0.1  1.777906e+09         50   TUNNEL  0.730952
3   dns1.pcapng   127.0.0.1  1.777906e+09         48   TUNNEL  0.865954
4   dns1.pcapng   127.0.0.1  1.777906e+09         46   TUNNEL  0.729563
5   dns1.pcapng   127.0.0.1  1.777906e+09         48   TUNNEL  0.808308
6   dns1.pcapng   127.0.0.1  1.777906e+09         45   TUNNEL  0.860907
7   dns1.pcapng   127.0.0.1  1.777906e+09        197   TUNNEL  0.799255
8   dns1.pcapng   127.0.0.1  1.777906e+09         27   TUNNEL  0.850817
9   dns1.pcapng   127.0.0.1  1.777906e+09         23   TUNNEL  0.671405
10  dns1.pcapng   127.0.0.1  1.777906e+09         53   TUNNEL  0.887994
11  dns1.pcapng   127.0.0.1  1.777906e+09         15   TUNNEL  0.839905
12  dns1.pcapng   127.0.0.1  1.777906e+09         17   TUNNEL  0

## 4. Summary table

In [6]:
base_learner_cols = [c for c in df_all.columns if c.startswith('p_tunnel_')]

summary_cols = ['pcap', 'hint', 'src_ip', 'window_start', 'n_packets',
                'decision', 'p_tunnel'] + base_learner_cols

print('=== CLASSIFICATION RESULTS ===')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.4f}'.format)
display(df_all[summary_cols].sort_values(['pcap', 'src_ip', 'window_start']))

=== CLASSIFICATION RESULTS ===

,pcap,hint,src_ip,window_start,n_packets,decision,p_tunnel,p_tunnel_lr,p_tunnel_rf,p_tunnel_gb
0,dns1.pcapng,dns1,127.0.0.1,1777906113.7659,37,TUNNEL,0.7370,0.4578,0.6147,0.9988
1,dns1.pcapng,dns1,127.0.0.1,1777906123.7659,47,BENIGN,0.3424,0.7397,0.4860,0.0001
2,dns1.pcapng,dns1,127.0.0.1,1777906133.7659,50,TUNNEL,0.7310,0.4236,0.6168,0.9988
3,dns1.pcapng,dns1,127.0.0.1,1777906143.7659,48,TUNNEL,0.8660,0.8260,0.7531,0.9988
4,dns1.pcapng,dns1,127.0.0.1,1777906153.7659,46,TUNNEL,0.7296,0.4217,0.6143,0.9988
5,dns1.pcapng,dns1,127.0.0.1,1777906163.7659,48,TUNNEL,0.8083,0.7871,0.6285,0.9988
6,dns1.pcapng,dns1,127.0.0.1,1777906173.7659,45,TUNNEL,0.8609,0.7986,0.7542,0.9988
7,dns1.pcapng,dns1,127.0.0.1,1777906183.7659,197,TUNNEL,0.7993,0.9978,0.5005,0.9988
8,dns1.pcapng,dns1,127.0.0.1,1777906193.7659,27,TUNNEL,0.8508,0.7042,0.7762,0.9988
9,dns1.pcapng,dns1,127.0.0.1,1777906203.7659,23,TUNNEL,0.6714,0.2537,0.5529,0.9988


In [7]:
# Per-file verdict summary
print('=== PER-FILE SUMMARY ===')
for pcap_name, grp in df_all.groupby('pcap'):
    n_total  = len(grp)
    n_tunnel = (grp['decision'] == 'TUNNEL').sum()
    n_benign = (grp['decision'] == 'BENIGN').sum()
    avg_p    = grp['p_tunnel'].mean()
    print(f'{pcap_name}:')
    print(f'  Windows : {n_total}  |  TUNNEL: {n_tunnel}  |  BENIGN: {n_benign}')
    print(f'  avg P(tunnel): {avg_p:.4f}')
    print()

=== PER-FILE SUMMARY ===

dns1.pcapng:

  Windows : 54  |  TUNNEL: 21  |  BENIGN: 33

  avg P(tunnel): 0.3809

dns2.pcapng:

  Windows : 6  |  TUNNEL: 1  |  BENIGN: 5

  avg P(tunnel): 0.1680

## 5. Feature-level analysis — what drove each decision?

In [8]:
# Key discriminative features from training analysis
KEY_FEATURES = [
    'query_rate', 'unique_qnames', 'unique_qname_ratio',
    'avg_qname_len', 'avg_subdomain_len', 'avg_max_label_len',
    'entropy_mean', 'subdomain_entropy_mean',
    'avg_b64_ratio', 'avg_hex_ratio', 'avg_unique_char_ratio',
    'tunnel_type_frac', 'txt_frac', 'null_frac',
    'payload_mean', 'payload_max',
    'avg_rdata_len', 'avg_answer_count',
    'base_entropy', 'base_gini',
    'tcp_frac', 'nxdomain_frac',
]

present = [f for f in KEY_FEATURES if f in df_all.columns]

print('=== KEY FEATURE VALUES PER WINDOW ===')
display(df_all[['pcap', 'src_ip', 'decision', 'p_tunnel'] + present]
        .sort_values(['pcap', 'src_ip'])
        .reset_index(drop=True))

=== KEY FEATURE VALUES PER WINDOW ===

,pcap,src_ip,decision,p_tunnel,query_rate,unique_qnames,unique_qname_ratio,avg_qname_len,avg_subdomain_len,avg_max_label_len,...,txt_frac,null_frac,payload_mean,payload_max,avg_rdata_len,avg_answer_count,base_entropy,base_gini,tcp_frac,nxdomain_frac
0,dns1.pcapng,127.0.0.1,TUNNEL,0.7370,4.2691,9,0.2432,23.5135,11.7568,8.6486,...,0.0000,0.0000,46.8649,69.0000,0.0000,0.0000,1.8274,0.6852,0.0000,0.0000
1,dns1.pcapng,127.0.0.1,BENIGN,0.3424,4.7318,12,0.2553,22.1277,9.8511,8.5532,...,0.0000,0.0000,45.9787,69.0000,0.0000,0.0000,2.4736,0.7777,0.0000,0.0000
2,dns1.pcapng,127.0.0.1,TUNNEL,0.7310,5.5809,13,0.2600,24.1200,12.6400,8.8800,...,0.0000,0.0000,50.0400,69.0000,0.0000,0.0000,2.6080,0.8000,0.0000,0.0000
3,dns1.pcapng,127.0.0.1,TUNNEL,0.8660,4.9526,16,0.3333,26.5833,15.3333,9.2917,...,0.0000,0.0000,53.7500,72.0000,0.0000,0.0000,2.3680,0.7457,0.0000,0.0000
4,dns1.pcapng,127.0.0.1,TUNNEL,0.7296,5.3359,16,0.3478,24.6739,14.3913,8.6739,...,0.0000,0.0000,50.3261,72.0000,0.0000,0.0000,2.2390,0.7391,0.0000,0.0000
5,dns1.pcapng,127.0.0.1,TUNNEL,0.8083,4.9822,13,0.2708,24.6667,13.1875,9.2708,...,0.0000,0.0000,50.0000,72.0000,0.0000,0.0000,2.2526,0.7535,0.0000,0.0000
6,dns1.pcapng,127.0.0.1,TUNNEL,0.8609,4.6914,12,0.2667,25.6444,15.2889,9.5556,...,0.0000,0.0000,51.7111,71.0000,0.0000,0.0000,2.2409,0.7477,0.0000,0.0000
7,dns1.pcapng,127.0.0.1,TUNNEL,0.7993,20.3196,20,0.1015,25.0000,13.4721,9.3807,...,0.0000,0.0000,80.0711,106.0000,6.2284,0.7919,2.5594,0.7851,0.0000,0.2893
8,dns1.pcapng,127.0.0.1,TUNNEL,0.8508,2.9351,7,0.2593,27.7037,15.8148,11.3704,...,0.0000,0.0000,51.8148,71.0000,0.0000,0.0000,1.8157,0.6941,0.0000,0.0000
9,dns1.pcapng,127.0.0.1,TUNNEL,0.6714,2.7271,8,0.3478,26.8261,15.4348,11.0870,...,0.0000,0.0000,52.9565,72.0000,0.0000,0.0000,2.1084,0.7372,0.0000,0.0000


In [9]:
# Compare tunnel vs benign windows (if both exist)
if df_all['decision'].nunique() == 2:
    print('=== TUNNEL vs BENIGN FEATURE MEANS ===')
    grp = df_all.groupby('decision')[present].mean().T
    grp['delta'] = (grp.get('TUNNEL', 0) - grp.get('BENIGN', 0)).abs()
    display(grp.sort_values('delta', ascending=False).head(20))
else:
    verdict = df_all['decision'].iloc[0]
    print(f'All windows classified as {verdict} — no contrast available.')
    print('Feature means for these windows:')
    display(df_all[present].mean().sort_values(ascending=False).head(15))

=== TUNNEL vs BENIGN FEATURE MEANS ===

decision,BENIGN,TUNNEL,delta
query_rate,288.4197,428.2878,139.8681
payload_max,118.4474,110.9091,7.5383
payload_mean,82.9078,76.1527,6.7550
avg_subdomain_len,8.6064,14.2264,5.6200
avg_rdata_len,8.6486,3.6903,4.9583
unique_qnames,3.2368,7.7273,4.4904
avg_qname_len,21.6886,25.7774,4.0887
subdomain_entropy_mean,2.2360,2.8869,0.6509
base_entropy,1.0228,1.5654,0.5426
avg_answer_count,1.1030,0.7644,0.3386


## 6. Low-confidence windows (model uncertainty)

In [10]:
uncertain = df_all[(df_all['p_tunnel'] > 0.3) & (df_all['p_tunnel'] < 0.7)].copy()

if len(uncertain) == 0:
    print('No uncertain windows (all predictions strongly confident). Good.')
else:
    print(f'{len(uncertain)} uncertain windows (0.3 < P(tunnel) < 0.7):')
    display(uncertain[['pcap', 'src_ip', 'window_start', 'n_packets',
                        'decision', 'p_tunnel'] + present[:10]])

7 uncertain windows (0.3 < P(tunnel) < 0.7):

,pcap,src_ip,window_start,n_packets,decision,p_tunnel,query_rate,unique_qnames,unique_qname_ratio,avg_qname_len,avg_subdomain_len,avg_max_label_len,entropy_mean,subdomain_entropy_mean,avg_b64_ratio,avg_hex_ratio
1,dns1.pcapng,127.0.0.1,1777906123.7659,47,BENIGN,0.3424,4.7318,12,0.2553,22.1277,9.8511,8.5532,3.6588,2.5261,0.8166,0.3015
9,dns1.pcapng,127.0.0.1,1777906203.7659,23,TUNNEL,0.6714,2.7271,8,0.3478,26.8261,15.4348,11.0870,3.7291,2.9197,0.8134,0.2821
22,dns1.pcapng,127.0.0.1,1777906383.7659,6,BENIGN,0.4744,0.9538,1,0.1667,21.0000,8.0000,8.0000,3.6538,2.7500,0.8750,0.3750
33,dns1.pcapng,127.0.0.53,1777906186.6810,129,BENIGN,0.3351,18.9558,15,0.1163,23.5969,11.5581,9.4419,3.6904,2.4626,0.7881,0.2908
42,dns1.pcapng,127.0.0.53,1777906326.6810,4,TUNNEL,0.5074,4.6437,2,0.5000,21.5000,8.5000,8.0000,3.6758,2.8489,0.8819,0.3542
45,dns1.pcapng,127.0.0.53,1777906376.6810,4,TUNNEL,0.5812,6592.2263,1,0.2500,21.0000,8.0000,8.0000,3.6538,2.7500,0.8750,0.3750
55,dns2.pcapng,127.0.0.1,1777906566.6358,12,TUNNEL,0.6921,2.3979,5,0.4167,27.8333,15.1667,11.5000,3.7244,3.0092,0.9259,0.2574


## 7. CLI tool fix — feature mismatch patch

In [11]:
print('=== CLI TOOL FIX REQUIRED ===')
print()
print('File: DNS_tunnelling_CLI/feature_extraction.py')
print()
print('Problem:')
print('  EXPECTED_FEATURES is missing "base_entropy" and "base_gini".')
print('  inference.py raises ValueError on model load because these features')
print('  are in the model joblib but not in EXPECTED_FEATURES.')
print()
print('Fix A — add to EXPECTED_FEATURES (after "tunnel_type_frac"):')
print()
print('  OLD order (after tunnel_type_frac):')
print('    "tcp_frac", "avg_qname_len", ...')
print()
print('  NEW order:')
print('    "base_entropy", "base_gini", "tcp_frac", "avg_qname_len", ...')
print()
print('Fix B — add computation inside compute_features():')
print()
print('  from collections import Counter')
print('  import math')
print()
print('  def _shannon_entropy_counts(counts):')
print('      n = sum(counts.values())')
print('      if not n: return 0.0')
print('      return -sum((c/n)*math.log2(c/n) for c in counts.values() if c)')
print()
print('  def _gini_counts(counts):')
print('      n = sum(counts.values())')
print('      if not n: return 0.0')
print('      return 1.0 - sum((c/n)**2 for c in counts.values())')
print()
print('  # Inside compute_features(), after extracting qnames:')
print('  base_cnt = Counter("."..join(q.split(".")[-2:]) for q in qnames if q)')
print('  features["base_entropy"] = _shannon_entropy_counts(base_cnt)')
print('  features["base_gini"]    = _gini_counts(base_cnt)')

=== CLI TOOL FIX REQUIRED ===

File: DNS_tunnelling_CLI/feature_extraction.py

Problem:

  EXPECTED_FEATURES is missing "base_entropy" and "base_gini".

  inference.py raises ValueError on model load because these features

...

      return 1.0 - sum((c/n)**2 for c in counts.values())

  # Inside compute_features(), after extracting qnames:

  base_cnt = Counter("."..join(q.split(".")[-2:]) for q in qnames if q)

  features["base_entropy"] = _shannon_entropy_counts(base_cnt)

  features["base_gini"]    = _gini_counts(base_cnt)

## 8. Analysis summary & recommendations

In [12]:
print('=== FINAL ANALYSIS ===')
print()

for pcap_name, grp in df_all.groupby('pcap'):
    n          = len(grp)
    n_tunnel   = (grp['decision'] == 'TUNNEL').sum()
    avg_p      = grp['p_tunnel'].mean()
    confident  = ((grp['p_tunnel'] > 0.9) | (grp['p_tunnel'] < 0.1)).sum()

    print(f'{pcap_name}')
    print(f'  Decision distribution : {n_tunnel}/{n} TUNNEL, {n-n_tunnel}/{n} BENIGN')
    print(f'  Mean P(tunnel)        : {avg_p:.4f}')
    print(f'  Confident windows     : {confident}/{n} (p>0.9 or p<0.1)')

    avg_qlen = grp['avg_qname_len'].mean() if 'avg_qname_len' in grp else 'N/A'
    avg_ent  = grp['entropy_mean'].mean()  if 'entropy_mean'  in grp else 'N/A'
    avg_b64  = grp['avg_b64_ratio'].mean() if 'avg_b64_ratio' in grp else 'N/A'

    print(f'  avg_qname_len         : {avg_qlen:.2f}' if isinstance(avg_qlen, float) else f'  avg_qname_len: {avg_qlen}')
    print(f'  entropy_mean          : {avg_ent:.4f}'  if isinstance(avg_ent,  float) else f'  entropy_mean: {avg_ent}')
    print(f'  avg_b64_ratio         : {avg_b64:.4f}'  if isinstance(avg_b64,  float) else f'  avg_b64_ratio: {avg_b64}')
    print()

print()
print('ISSUES TO FIX / INVESTIGATE:')
print()
print('1. CLI TOOL BUG (critical)')
print('   feature_extraction.py is missing base_entropy + base_gini.')
print('   CLI tool crashes on model load. Patch required (see Cell 7).')
print()
print('2. LAB-ENVIRONMENT BIAS (expected)')
print('   The model was trained on lab pcaps, so real traffic may show different')
print('   base entropy / gini profiles. If real benign traffic is misclassified,')
print('   threshold tuning (--threshold 0.7) is the first lever to pull.')
print()
print('3. WINDOW SIZE SENSITIVITY')
print('   If your generated traffic has very few packets per 10s window,')
print('   features like query_rate and entropy will be unreliable.')
print('   Check n_packets per window — windows with < 5 packets are unreliable.')
print()
print('4. PCAPNG vs PCAP')
print('   Scapy reads pcapng fine, but some fields (e.g. nanosecond timestamps)')
print('   may differ. If timestamp resolution looks off, normalise to seconds.')

=== FINAL ANALYSIS ===

dns1.pcapng

  Decision distribution : 21/54 TUNNEL, 33/54 BENIGN

  Mean P(tunnel)        : 0.3809

  Confident windows     : 18/54 (p>0.9 or p<0.1)

...

   features like query_rate and entropy will be unreliable.

   Check n_packets per window — windows with < 5 packets are unreliable.

4. PCAPNG vs PCAP

   Scapy reads pcapng fine, but some fields (e.g. nanosecond timestamps)

   may differ. If timestamp resolution looks off, normalise to seconds.